# 04b – Fine-tune Stable Diffusion 1.5 với LoRA: Tranh dân gian Đông Hồ (TV3 – Phase 3)

Notebook này thiết kế để chạy trực tiếp trên **Kaggle (GPU T4 16GB VRAM, Internet ON)** với bộ dữ liệu **Tranh dân gian Đông Hồ** (`dminh2603/dong-ho`).

Các bước thực hiện:
1. Kiểm tra GPU và thiết lập cache HuggingFace vào `/tmp/hf_cache` để không làm đầy `/kaggle/working/`.
2. Clone/cập nhật nhánh `Diffusion` của repository và cài bổ sung thư viện còn thiếu (`diffusers`, `peft`, `accelerate`) mà không ghi đè PyTorch CUDA sẵn có.
3. Tự động tìm dataset Đông Hồ (Kaggle input hoặc local fallback) và hiển thị lưới ảnh mẫu.
4. Chạy `python -m src.training.train_lora_sd --config configs/lora_sd.yaml` (training FP16 an toàn VRAM + sinh ảnh đối chứng Base SD vs. LoRA SD cùng seed).
5. Hiển thị log tóm tắt, loss curve và ảnh đối chứng `base_vs_lora.png`.
6. Đóng gói checkpoint LoRA và toàn bộ outputs thành file zip để tải về.

## 1. Kiểm tra môi trường GPU & cấu hình Cache HuggingFace

In [ ]:
import os
import torch

# Lưu cache model (~4-5 GB) ở /tmp để tránh đầy ổ /kaggle/working/
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

!nvidia-smi
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
print("HF_HOME:", os.environ["HF_HOME"])


## 2. Clone / Cập nhật Repository & cài thư viện bổ sung

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Kienhtht1/DeepLearning_BTL.git"
BRANCH = "Diffusion"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK_ROOT / "DeepLearning_BTL"

if (REPO_DIR / ".git").exists():
    print(f"Repo đã tồn tại tại {REPO_DIR}, tiến hành cập nhật nhánh {BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print(f"Clone nhánh {BRANCH} về {REPO_DIR}...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

%cd {REPO_DIR}


In [ ]:
# Chỉ cài bổ sung các gói còn thiếu, KHÔNG ghi đè torch/torchvision CUDA sẵn có trên Kaggle
!pip install -q diffusers peft accelerate

import accelerate
import diffusers
import peft
import torch
import torchvision
import transformers

print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("diffusers   :", diffusers.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)


## 3. Tự động định vị bộ ảnh Đông Hồ & xem trước mẫu

In [ ]:
import math
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from src.data.custom_dataset import IMG_EXTS

CANDIDATE_DIRS = [
    Path("/kaggle/input/dong-ho"),
    Path("/kaggle/input/datasets/dminh2603/dong-ho"),
    Path("data/custom/tranh_dan_gian"),
]

def find_image_files(folder: Path) -> list[Path]:
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and not p.name.startswith(".") and p.suffix.lower() in IMG_EXTS
    )

DETECTED_DATA_DIR = None
sample_paths = []
for cand in CANDIDATE_DIRS:
    if cand.exists():
        imgs = find_image_files(cand)
        if imgs:
            DETECTED_DATA_DIR = cand
            sample_paths = imgs
            break

if DETECTED_DATA_DIR is None:
    raise FileNotFoundError(
        f"Không tìm thấy ảnh hợp lệ trong các thư mục: {[str(c) for c in CANDIDATE_DIRS]}"
    )

print(f"Dataset phát hiện tại : {DETECTED_DATA_DIR}")
print(f"Tổng số file ảnh     : {len(sample_paths)}")

# Hiển thị lưới 8 bức tranh Đông Hồ mẫu
n_show = min(8, len(sample_paths))
cols = 4
rows = math.ceil(n_show / cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 3.2 * rows), squeeze=False)
for idx in range(rows * cols):
    ax = axes[idx // cols, idx % cols]
    ax.axis("off")
    if idx < n_show:
        with Image.open(sample_paths[idx]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(sample_paths[idx].name, fontsize=9)
plt.suptitle("Mẫu dữ liệu Tranh dân gian Đông Hồ", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Chạy Fine-tune LoRA Stable Diffusion 1.5 & Sinh ảnh đối chứng

In [ ]:
!python -m src.training.train_lora_sd --config configs/lora_sd.yaml --data-dir "{DETECTED_DATA_DIR}"


## 5. Kết quả huấn luyện, Loss Curve & So sánh Base SD vs. LoRA SD

In [ ]:
import json
from pathlib import Path
from IPython.display import Image as IPyImage, display

log_path = Path("outputs/lora_sd_train_log.json")
curve_path = Path("outputs/lora_sd_loss_curve.png")
comp_path = Path("outputs/samples/lora_sd/base_vs_lora.png")

if log_path.is_file():
    log_data = json.loads(log_path.read_text(encoding="utf-8"))
    print("=== TÓM TẮT HUẤN LUYỆN LORA STABLE DIFFUSION 1.5 ===")
    print(f"- Trạng thái          : {log_data.get('status')}")
    print(f"- Số ảnh train        : {log_data.get('num_images')}")
    print(f"- Trainable LoRA      : {log_data.get('trainable_params'):,} / {log_data.get('total_unet_params'):,} ({log_data.get('trainable_ratio_pct')}%)")
    print(f"- Số step hoàn thành  : {log_data.get('completed_steps')} / {log_data.get('max_train_steps')}")
    print(f"- Tổng thời gian      : {log_data.get('total_time_sec')} giây")
    if log_data.get("history"):
        print(f"- Loss bước cuối      : {log_data['history'][-1]['loss']}")

if curve_path.is_file():
    display(IPyImage(filename=str(curve_path)))

if comp_path.is_file():
    display(IPyImage(filename=str(comp_path)))


## 6. Đóng gói kết quả tải về máy

In [ ]:
from pathlib import Path
import zipfile

out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
zip_path = out_root / "lora_sd_results.zip"

targets = [
    Path("checkpoints/lora_sd"),
    Path("outputs/lora_sd_train_log.json"),
    Path("outputs/lora_sd_loss_curve.png"),
    Path("outputs/samples/lora_sd"),
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for target in targets:
        if target.is_file():
            zf.write(target, arcname=target.as_posix())
        elif target.is_dir():
            for file_path in sorted(target.rglob("*")):
                if file_path.is_file():
                    zf.write(file_path, arcname=file_path.as_posix())

size_mb = zip_path.stat().st_size / (1024 ** 2)
print(f"Đã đóng gói thành công: {zip_path} ({size_mb:.2f} MB)")
